# Parte 5 — Otimização de hiperparâmetros e generalização

A Random Forest venceu a comparação da Parte 4. Agora ajustamos os
hiperparâmetros dela com **Grid Search + cross-validation** (ainda só no
treino), e só **no final** desta parte tocamos o conjunto de teste —
pela primeira e única vez em todo o projeto — para checar se o modelo
generaliza ou só decorou o treino (overfitting).


## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
)
import joblib

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.utils.paths import GOLD_DIR
from src.preprocessing.pipeline import build_preprocessor, COLUNAS_CATEGORICAS, COLUNAS_NUMERICAS

PASTA_MODELO = GOLD_DIR.parent / "model_input"


## 2. Carregar treino e teste

O teste é carregado aqui, mas **não entra em nenhum cálculo até a seção
6** — só o guardamos pronto.


In [2]:
train = pd.read_parquet(PASTA_MODELO / "train.parquet")
test = pd.read_parquet(PASTA_MODELO / "test.parquet")

X_train = train[COLUNAS_CATEGORICAS + COLUNAS_NUMERICAS]
y_train = train["alfabetizado_flag"]

X_test = test[COLUNAS_CATEGORICAS + COLUNAS_NUMERICAS]
y_test = test["alfabetizado_flag"]

print(f"Treino: {len(X_train):,} | Teste: {len(X_test):,}")


Treino: 3,094,399 | Teste: 773,600


## 3. Grid de hiperparâmetros

Mantemos o grid enxuto de propósito: 4 combinações (`n_estimators` x
`max_depth`) com `cv=3` = 12 treinos de Random Forest no total. Num grid
maior (mais parâmetros, mais valores), o tempo cresce rápido — com 3,8
milhões de linhas, cada combinação a mais custa minutos reais, não
segundos. Isso já é suficiente para demonstrar a técnica e melhorar sobre
o baseline da Parte 4.

**Aviso de tempo:** isso pode demorar bastante (depende da sua máquina)
— é esperado, não é travamento.


In [3]:
pipeline_base = Pipeline(steps=[
    ("preprocessor", build_preprocessor()),
    ("modelo", RandomForestClassifier(random_state=42, n_jobs=-1)),
])

param_grid = {
    "modelo__n_estimators": [100, 200],
    "modelo__max_depth": [15, None],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipeline_base,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=1,        # o paralelismo ja acontece dentro da Random Forest (n_jobs=-1 acima)
    refit=True,       # ao final, reajusta automaticamente o melhor modelo em TODO o treino
    return_train_score=True,
)

grid_search.fit(X_train, y_train)
print("Concluído.")


Concluído.


## 4. Melhor combinação encontrada

In [4]:
print("Melhores hiperparâmetros:", grid_search.best_params_)
print(f"Melhor ROC-AUC (média em cross-validation): {grid_search.best_score_:.4f}")


Melhores hiperparâmetros: {'modelo__max_depth': 15, 'modelo__n_estimators': 200}
Melhor ROC-AUC (média em cross-validation): 0.7684


In [5]:
resultados_grid = pd.DataFrame(grid_search.cv_results_)[[
    "param_modelo__n_estimators", "param_modelo__max_depth",
    "mean_train_score", "mean_test_score", "std_test_score",
]].sort_values("mean_test_score", ascending=False)
resultados_grid.columns = ["n_estimators", "max_depth", "roc_auc_treino", "roc_auc_validacao", "desvio_padrao"]
resultados_grid


,n_estimators,max_depth,roc_auc_treino,roc_auc_validacao,desvio_padrao
1,200,15,0.774080,0.768438,0.000311
0,100,15,0.774119,0.768424,0.000275
3,200,None,0.819282,0.759932,0.000450
2,100,None,0.819127,0.759753,0.000440


**Checagem de overfitting aqui mesmo:** compare `roc_auc_treino` com
`roc_auc_validacao` em cada linha. Se o treino estiver muito mais alto
que a validação (uma diferença grande, tipo 0.15+), é sinal de
overfitting — o modelo está decorando o treino em vez de aprender um
padrão generalizável. Se as duas colunas ficarem próximas, o modelo está
generalizando bem dentro do que o cross-validation consegue medir.


## 5. Melhorou sobre o baseline da Parte 4?

Comparação direta: ROC-AUC da Random Forest **antes** de otimizar
(Parte 4) vs **depois** do Grid Search.


In [6]:
roc_auc_antes = 0.7598  # valor obtido na Parte 4 -- ajuste se o seu tiver sido diferente
roc_auc_depois = grid_search.best_score_

print(f"Antes da otimização: {roc_auc_antes:.4f}")
print(f"Depois da otimização: {roc_auc_depois:.4f}")
print(f"Diferença: {roc_auc_depois - roc_auc_antes:+.4f}")


Antes da otimização: 0.7598
Depois da otimização: 0.7684
Diferença: +0.0086


## 6. A ÚNICA avaliação no conjunto de teste

Primeira e última vez que o teste é usado neste projeto. O modelo final
(`grid_search.best_estimator_`, já reajustado em 100% do treino pelo
`refit=True`) nunca viu essas linhas.


In [7]:
melhor_pipeline = grid_search.best_estimator_

y_pred = melhor_pipeline.predict(X_test)
y_proba = melhor_pipeline.predict_proba(X_test)[:, 1]

metricas_teste = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_proba),
}
pd.Series(metricas_teste).round(4)


accuracy     0.6919
precision    0.6585
recall       0.8301
f1           0.7344
roc_auc      0.7679
dtype: float64

**Checagem final de generalização:** compare o `roc_auc` do teste com o
`roc_auc_validacao` da seção 4 (cross-validation, dentro do treino). Se
os dois números ficarem próximos, o modelo generaliza de verdade — o
desempenho que vimos durante o desenvolvimento não foi um acaso do
cross-validation, se confirma num dado que o modelo nunca viu.


In [8]:
print(classification_report(y_test, y_pred, target_names=["Não alfabetizado", "Alfabetizado"]))


                  precision    recall  f1-score   support

Não alfabetizado       0.75      0.55      0.63    376691
    Alfabetizado       0.66      0.83      0.73    396909

        accuracy                           0.69    773600
       macro avg       0.71      0.69      0.68    773600
    weighted avg       0.70      0.69      0.69    773600



In [9]:
matriz = confusion_matrix(y_test, y_pred)
pd.DataFrame(
    matriz,
    index=["Real: Não", "Real: Sim"],
    columns=["Previsto: Não", "Previsto: Sim"],
)


,Previsto: Não,Previsto: Sim
Real: Não,205805,170886
Real: Sim,67453,329456


## 7. Salvar o modelo final

Salvamos o pipeline completo (pré-processamento + modelo já treinado)
para a Parte 6 usar direto, sem precisar retreinar.


In [10]:
caminho_modelo = PASTA_MODELO / "melhor_modelo.joblib"
joblib.dump(melhor_pipeline, caminho_modelo)
print(f"Salvo em: {caminho_modelo}")


Salvo em: C:\Users\Guilherme\Desktop\Projeto\modelagem\data\model_input\melhor_modelo.joblib
